# Project Delphi (Merlin)

## 02 - Title Embeddings

### Overview:

In this notebook, we'll generate **semantic embeddings** for YouTube video titles using the `all-MiniLM-L6-v2` model from [Sentence Transformers](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). These embeddings will capture the *contextual meaning* of each title -- allowing our downstream XGBoost models to learn patterns beyond just word counts or keywords.  

In other words, we're transforming human language into a numeric representation that the model can understand.

---

### The Plan:
1. **Load** the cleaned dataset from `01_data_preparation`  
2. **Initialize** the Sentence Transformer model (`all-MiniLM-L6-v2`)  
3. **Encode** all video titles into 384-dimensional embeddings  
4. **Append** these embeddings to the main DataFrame  
5. **Save** the new feature-rich dataset for the next stage (`03_model_training_xgboost`)

---

### Why This Matters

Embedding the titles gives our model a way to:
- Understand *semantic similarity* (e.g., “49ers lose tough game” ≈ “San Francisco falls short”)
- Capture *emotional tone* and *narrative framing*
- Move beyond naive keyword matching and learn richer title-performance relationships

## Load the cleaned (and engineered) dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the cleaned and engineered dataset
import pandas as pd

data_path = '/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_clean.pkl'
df = pd.read_pickle(data_path)

# Check the first few rows of the DataFrame
df.head()

In [ ]:
# And confirm the shape -- should be (298, 6)
df.shape

## Initialize the Sentence Transformer model

For this step, we'll use **`all-MiniLM-L6-v2`**, a lightweight yet powerful transformer that turns each video title into **a 384-dimensional embedding** capturing its meaning and tone.

In [ ]:
# Initialize the Sentence Transformer model
from sentence_transformers import SentenceTransformer

# Load the pre-trained model, as speficied above
model = SentenceTransformer('all-MiniLM-L6-v2')
model

The `all-MiniLM-L6-v2` model is a streamlined version of BERT that compresses its knowlegde into a smaller 6-layer transformer and then applies a **pooling layer** to combine word-level embeddings into a single 384-dimensional vector capturing the overall meaning, tone, and context of each title.

## Generate the title embeddings

We'll encode each video title into a 384-dimensional vector using the model we just loaded.

In [ ]:
# Start by importing numpy for computations
import numpy as np

# Optional check: make sure the column 'titles' actually exists
# assert 'title' in df.columns, "Expected a 'title' column in the dataframe."
# titles = df['title'].astype(str).fillna('').tolist()

# Generate 384-dimensional embeddings for each video title using the pre-trained Sentence Transformer
# Important notes:
# - titles: This is the list of text strings to embed (each representing a video title)
# - batch_size=64: Small batches improve efficiency and avoid memory overload
# - show_progress_bar=True: This shows a progress bar for visibility during encoding
# - convert_to_numpy=True: This returns the embeddings as a NumPy array (easy to use later with ML models)
# - normalize_embeddings=True: This scales each embedding vector to have the same size, helpful for comparison
embeddings = model.encode(titles, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# Display the embeddings
embeddings

In [ ]:
# Check the shape of the embeddings
# Note: This should be (298, 384) becasue we have 298 title embeddings that are 384-dimensional
embeddings.shape

## Append the embeddings to the cleaned (and engineered) DataFrame

Now that we've generated the embeddings, we're ready to append them to our DataFrame (`df`). This way, every video title's semantic representation will be mapped onto its numeric features -- creating a unified dataset that blends the two types of input features.

In [ ]:
# Need to turn the embeddings into a DataFrame, as they are currently an array
# type(embeddings)

# Use the .DataFrame() function
# Note: The f-string + list comprehension create those column names we see below!
embeddings_df = pd.DataFrame(embeddings, columns=[f'embed_{i}' for i in range(embeddings.shape[1])])

# Check the first few rows
embeddings_df.head()

In [ ]:
# Now to combine or concatenate the two DataFrames
# Note: 1) .reset_index(drop=True) makes sure the two DataFrames match up row-by-row
# and 2) axix=1 tells pandas to add columns side-by-side instead of stacking rows
df_with_embeddings = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

# Check the results -- or at least the first few rows
df_with_embeddings.head()

In [ ]:
# And now check the shape of the new DataFrame
df_with_embeddings.shape

The above shape tells us we have 298 rows (videos) and 390 coluns (4 numeric features, 384 embedding dimensions and 2 target variables). That means we're ready to save the DataFrame (`df_with_embeddings`) and dive into the modeling.

## Save the feature-rich dataset (`df_with_embeddings`)

In [ ]:
# Save the feature-rich dataset for modeling
df_with_embeddings.to_pickle('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_with_embeddings.pkl')
df_with_embeddings.to_csv('/content/drive/MyDrive/Colab Notebooks/project_delphi/data/project_delphi_with_embeddings.csv', index=False)


In [ ]:
import sentence_transformers
sentence_transformers.__version__

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embedder.save("/content/merlin_embedder")
!zip -r /content/merlin_embedder.zip /content/merlin_embedder